In [33]:
from typing import Dict

import torch
import numpy as np
import lightning.pytorch as pl
from bayesian.bayesian_net import BayesianFCN
import pinnstorch

In [34]:
def read_data_fn(root_path):
    """Read and preprocess data from the specified root path.

    :param root_path: The root directory containing the data.
    :return: Processed data will be used in Mesh class.
    """

    data = pinnstorch.utils.load_data(root_path, "NLS.mat")
    exact = data["uu"]
    exact_u = np.real(exact) # N x T
    exact_v = np.imag(exact) # N x T
    exact_h = np.sqrt(exact_u**2 + exact_v**2) # N x T
    return {"u": exact_u, "v": exact_v, "h": exact_h}

In [35]:
time_domain = pinnstorch.data.TimeDomain(t_interval=[0, 1.57079633], t_points = 201)
spatial_domain = pinnstorch.data.Interval(x_interval= [-5, 4.9609375], shape = [256, 1])

In [36]:
mesh = pinnstorch.data.Mesh(root_dir='../data',
                            read_data_fn=read_data_fn,
                            spatial_domain = spatial_domain,
                            time_domain = time_domain)

In [37]:
def read_data_fn(root_path):
    """Read and preprocess data from the specified root path.

    :param root_path: The root directory containing the data.
    :return: Processed data will be used in PointCloud class.
    """

    data = pinnstorch.utils.load_data(root_path, "NLS.mat")

    x = data["x"].T  # N x 1
    t = data["tt"].T  # T x 1
    
    exact = data["uu"]
    exact_u = np.real(exact) # N x T
    exact_v = np.imag(exact) # N x T
    exact_h = np.sqrt(exact_u**2 + exact_v**2) # N x T
    
    return pinnstorch.data.PointCloudData(
            spatial=[x], time=[t], solution={"u": exact_u, "v": exact_v, "h": exact_h}
    )

In [38]:
mesh = pinnstorch.data.PointCloud(root_dir='./data',
                                  read_data_fn=read_data_fn)

In [39]:
N0 = 50
in_c = pinnstorch.data.InitialCondition(mesh = mesh,
                                        num_sample = N0,
                                        solution = ['u', 'v'])


In [40]:
N_b = 50
pe_b = pinnstorch.data.PeriodicBoundaryCondition(mesh = mesh,
                                                 num_sample = 50,
                                                 derivative_order = 1,
                                                 solution = ['u', 'v'])

In [41]:
N_f = 20000
me_s = pinnstorch.data.MeshSampler(mesh = mesh,
                                   num_sample = N_f,
                                   collection_points = ['f_v', 'f_u'])

In [42]:
val_s = pinnstorch.data.MeshSampler(mesh = mesh,
                                    solution = ['u', 'v', 'h'])

In [47]:
net = BayesianFCN(layers = [2, 100, 100, 100, 100, 2],
                            output_names = ['u', 'v'],
                            lb=mesh.lb,
                            ub=mesh.ub)

In [48]:
def output_fn(outputs: Dict[str, torch.Tensor],
              x: torch.Tensor,
              t: torch.Tensor):
    """Define `output_fn` function that will be applied to outputs of net."""

    outputs["h"] = torch.sqrt(outputs["u"] ** 2 + outputs["v"] ** 2)

    return outputs

In [49]:
def pde_fn(outputs: Dict[str, torch.Tensor],
           x: torch.Tensor,
           t: torch.Tensor):   
    """Define the partial differential equations (PDEs)."""
    u_x, u_t = pinnstorch.utils.gradient(outputs["u"], [x, t])
    v_x, v_t = pinnstorch.utils.gradient(outputs["v"], [x, t])

    u_xx = pinnstorch.utils.gradient(u_x, x)[0]
    v_xx = pinnstorch.utils.gradient(v_x, x)[0]

    outputs["f_u"] = u_t + 0.5 * v_xx + (outputs["u"] ** 2 + outputs["v"] ** 2) * outputs["v"]
    outputs["f_v"] = v_t - 0.5 * u_xx - (outputs["u"] ** 2 + outputs["v"] ** 2) * outputs["u"]

    return outputs

In [50]:
train_datasets = [me_s, in_c, pe_b]
val_dataset = val_s
datamodule = pinnstorch.data.PINNDataModule(train_datasets = [me_s, in_c, pe_b],
                                            val_dataset = val_dataset,
                                            pred_dataset = val_s)

In [30]:
from typing import Dict

import torch
import numpy as np
from lightning.pytorch.callbacks import ModelCheckpoint
import lightning.pytorch as pl

from lightning.pytorch.loggers import WandbLogger

import pinnstorch

wandb_logger = WandbLogger(
    entity="viriyadhika1",
    project="pinn-lab1",
    name=f"Bayesian"
)
checkpoint_cb = ModelCheckpoint(
    dirpath=f"checkpoints/bayesian/",
    filename="pinn-{epoch:04d}-{val_loss:.4f}",
    monitor="val/loss",
    mode="min",
    save_top_k=1,       # best model
    save_last=True,     # <-- always save last.ckpt
    every_n_epochs=1000
)
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=100,
    logger=wandb_logger,
    callbacks=[checkpoint_cb]
)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [31]:
model = pinnstorch.models.PINNModule(net = net,
                                     pde_fn = pde_fn,
                                     output_fn = output_fn,
                                     loss_fn = 'mse')

In [32]:
trainer.fit(model=model, datamodule=datamodule)

wandb: Currently logged in as: viriyadhika-putra (viriyadhika1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name          | Type        | Params | Mode 
------------------------------------------------------
0 | net           | BayesianFCN | 61.6 K | train
1 | train_loss    | MeanMetric  | 0      | train
2 | val_loss      | MeanMetric  | 0      | train
3 | val_error     | MeanMetric  | 0      | train
4 | test_loss     | MeanMetric  | 0      | train
5 | test_error    | MeanMetric  | 0      | train
6 | val_loss_best | MinMetric   | 0      | train
------------------------------------------------------
61.6 K    Trainable params
0         Non-trainable params
61.6 K    Total params
0.246     Total estimated model params size (MB)
17        Modules in train mode
0         Modules in eval mode


/Users/viriyadhika/Documents/5. UofT/Courses/CSC2539H-PINN/pinns-torch/.venv/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/viriyadhika/Documents/5. UofT/Courses/CSC2539H-PINN/pinns-torch/.venv/lib/python3.11/site-packages/torch/jit/_trace.py:1310: TracerWarning: Trace had nondeterministic nodes. Did you forget call .eval() on your model? Nodes:
	%weight_eps : Float(100, 2, strides=[2, 1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_functions_forward_self_modules_net_modules_model_modules_input_parameters_weight_mu_, %40, %41, %42, %43, %44) # <eval_with_key>.19:35:0
	%bias_eps : Float(100, strides=[1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_functions_forward_self_modules_net_modules_model_modules_input_parameters_bias_mu_, %46, %47, %48, %49, %50) # <eval_with_key>.19:36:0
	%weight_eps_1 : Float(100, 100, strides=[100, 1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_functions_forward_self_modules_net_modules_model_modules_hidden_2_parameters_weight_mu_, %66, %67, %68, %69, %70) # <eval_with_key>.19:47:0
	%bias_eps_1 : Float(100, strides=[1], requires_gr

...

/Users/viriyadhika/Documents/5. UofT/Courses/CSC2539H-PINN/pinns-torch/.venv/lib/python3.11/site-packages/torch/jit/_trace.py:1310: TracerWarning: Trace had nondeterministic nodes. Did you forget call .eval() on your model? Nodes:
	%weight_eps : Float(100, 2, strides=[2, 1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_modules_net_modules_model_modules_input_parameters_weight_mu_, %40, %41, %42, %43, %44) # <eval_with_key>.27:35:0
	%bias_eps : Float(100, strides=[1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_modules_net_modules_model_modules_input_parameters_bias_mu_, %46, %47, %48, %49, %50) # <eval_with_key>.27:36:0
	%weight_eps_1 : Float(100, 100, strides=[100, 1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_modules_net_modules_model_modules_hidden_2_parameters_weight_mu_, %66, %67, %68, %69, %70) # <eval_with_key>.27:47:0
	%bias_eps_1 : Float(100, strides=[1], requires_grad=0, device=mps:0) = aten::randn_like(%L_self_modules_net_modules_mo

Epoch 99: 100%|██████████| 1/1 [00:00<00:00,  8.16it/s, v_num=ebds, train/loss_step=2.94e+3, val/loss=11.90, val/error_u=1.780, val/error_v=4.170, val/error_h=2.530, val/loss_best=2.830, train/loss_epoch=2.94e+3]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 1/1 [00:00<00:00,  7.29it/s, v_num=ebds, train/loss_step=2.94e+3, val/loss=11.90, val/error_u=1.780, val/error_v=4.170, val/error_h=2.530, val/loss_best=2.830, train/loss_epoch=2.94e+3]


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.
